# Lab 5.5: Agent SDK Hooks

**What you'll build:** A hook-based system for data normalization and compliance -- and learn why prompt-based normalization is unreliable.

**Estimated time:** 20-25 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way -- prompt instructions produce inconsistent date formats | 5 min |
| 2 | The Right Way -- PostToolUse hooks guarantee consistent normalization | 10 min |
| 3 | Your Turn -- build a PreToolCall hook for access control | 5 min |
| 4 | Stress Test -- verify hooks work across multiple tool calls | 5 min |

In [ ]:
import anthropic
import json
import re
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

## The Setup

You have a customer data tool that returns dates in various formats depending on the source system. Your downstream pipeline requires all dates in ISO 8601 format (YYYY-MM-DD). You also have a tool that returns monetary values in different formats.

The challenge: ensure 100% consistent formatting across all tool outputs.

In [ ]:
# Simulated tools that return inconsistent formats
CUSTOMER_RECORDS = {
    "C001": {
        "name": "Alice Johnson",
        "signup_date": "January 15, 2023",  # Long format
        "last_purchase": "03/22/2024",        # US format
        "subscription_renewal": "2024-12-01",  # ISO format
        "balance": "$1,234.56"
    },
    "C002": {
        "name": "Bob Smith",
        "signup_date": "22 Mar 2022",          # Day-first format
        "last_purchase": "11-15-2024",          # Dashed US format
        "subscription_renewal": "2025/01/15",   # Slash ISO-like
        "balance": "567.89"
    }
}

TOOLS = [
    {
        "name": "get_customer",
        "description": "Retrieve customer record by ID",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string"}
            },
            "required": ["customer_id"]
        }
    },
    {
        "name": "get_order_history",
        "description": "Get order history for a customer",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string"}
            },
            "required": ["customer_id"]
        }
    }
]

ORDER_HISTORY = {
    "C001": [
        {"order_id": "ORD-100", "date": "Mar 22, 2024", "total": "$89.99"},
        {"order_id": "ORD-101", "date": "12/01/2024", "total": "$24.99"}
    ],
    "C002": [
        {"order_id": "ORD-200", "date": "November 15, 2024", "total": "149.99"}
    ]
}

def execute_raw_tool(name, tool_input):
    """Execute tool returning RAW data (inconsistent formats)."""
    if name == "get_customer":
        record = CUSTOMER_RECORDS.get(tool_input["customer_id"])
        return json.dumps(record if record else {"error": "Not found"})
    elif name == "get_order_history":
        orders = ORDER_HISTORY.get(tool_input["customer_id"])
        return json.dumps(orders if orders else {"error": "No orders"})
    return json.dumps({"error": "Unknown tool"})

# Show the inconsistency
print("Raw data formats:")
for cid, record in CUSTOMER_RECORDS.items():
    print(f"  {cid}:")
    for key, val in record.items():
        if "date" in key.lower() or key in ["last_purchase", "subscription_renewal"]:
            print(f"    {key}: {val}")

---

## Phase 1: The Wrong Approach

Ask the model to normalize dates via prompt instructions.

In [ ]:
def run_agent_no_hooks(user_message):
    messages = [{"role": "user", "content": user_message}]
    system = """You are a data retrieval agent. IMPORTANT: Always normalize all dates to ISO 8601 format (YYYY-MM-DD) in your responses. Always format monetary values as plain numbers with 2 decimal places (no $ signs or commas)."""
    
    for _ in range(5):
        response = client.messages.create(
            model=MODEL, max_tokens=1024,
            system=system, tools=TOOLS, messages=messages
        )
        if response.stop_reason == "end_turn":
            return "".join(b.text for b in response.content if hasattr(b, "text"))
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = execute_raw_tool(block.name, block.input)  # Raw, unnormalized
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})
        if tool_results:
            messages.append({"role": "user", "content": tool_results})
    return "(cap)"

# Run 3 times and check consistency
print("Running 3 times with prompt-only normalization...\n")
for i in range(3):
    result = run_agent_no_hooks("Get customer C001's record and order history. List all dates and amounts.")
    print(f"Run {i+1}: {result[:200]}...\n")

print("Check: Are dates ALWAYS in YYYY-MM-DD format? Are amounts ALWAYS plain numbers?")
print("The model may normalize some dates but miss others, or be inconsistent across runs.")

---

## Phase 2: The Right Approach -- PostToolUse Hooks

A PostToolUse hook normalizes ALL tool output before the model sees it. This is deterministic.

In [ ]:
# Date normalization function
DATE_PATTERNS = [
    ("%B %d, %Y", None),          # January 15, 2023
    ("%b %d, %Y", None),          # Mar 22, 2024
    ("%d %b %Y", None),           # 22 Mar 2022
    ("%m/%d/%Y", None),           # 03/22/2024
    ("%m-%d-%Y", None),           # 11-15-2024
    ("%Y/%m/%d", None),           # 2025/01/15
    ("%Y-%m-%d", None),           # Already ISO
]

def normalize_date(date_str):
    """Convert any date format to ISO 8601 (YYYY-MM-DD)."""
    for fmt, _ in DATE_PATTERNS:
        try:
            dt = datetime.strptime(date_str.strip(), fmt)
            return dt.strftime("%Y-%m-%d")
        except ValueError:
            continue
    return date_str  # Return original if no pattern matches

def normalize_money(value_str):
    """Convert monetary string to plain decimal number."""
    if isinstance(value_str, (int, float)):
        return f"{value_str:.2f}"
    cleaned = re.sub(r'[\$,]', '', str(value_str))
    try:
        return f"{float(cleaned):.2f}"
    except ValueError:
        return value_str

def post_tool_use_hook(tool_name, tool_result_str):
    """
    PostToolUse hook: normalizes dates and monetary values
    in tool output BEFORE the model sees it.
    
    This is deterministic -- runs in code, not in the model.
    """
    try:
        data = json.loads(tool_result_str)
    except json.JSONDecodeError:
        return tool_result_str
    
    def normalize_value(key, value):
        if isinstance(value, str):
            # Normalize date-like fields
            if any(d in key.lower() for d in ["date", "renewal", "purchase", "created", "updated"]):
                return normalize_date(value)
            # Normalize money-like fields
            if any(m in key.lower() for m in ["balance", "total", "amount", "price", "cost"]):
                return normalize_money(value)
        return value
    
    def normalize_dict(d):
        if isinstance(d, dict):
            return {k: normalize_value(k, v) if not isinstance(v, (dict, list)) else normalize_dict(v) for k, v in d.items()}
        elif isinstance(d, list):
            return [normalize_dict(item) for item in d]
        return d
    
    normalized = normalize_dict(data)
    return json.dumps(normalized)


# Test the hook
raw = json.dumps(CUSTOMER_RECORDS["C001"])
normalized = post_tool_use_hook("get_customer", raw)

print("Before hook:")
print(json.dumps(json.loads(raw), indent=2))
print("\nAfter hook:")
print(json.dumps(json.loads(normalized), indent=2))

In [ ]:
def execute_with_hooks(name, tool_input):
    """Execute tool and apply PostToolUse hook."""
    raw_result = execute_raw_tool(name, tool_input)
    normalized_result = post_tool_use_hook(name, raw_result)
    return normalized_result

def run_agent_with_hooks(user_message):
    messages = [{"role": "user", "content": user_message}]
    # No normalization instructions needed in the system prompt!
    system = "You are a data retrieval agent. Retrieve and present customer information."
    
    for _ in range(5):
        response = client.messages.create(
            model=MODEL, max_tokens=1024,
            system=system, tools=TOOLS, messages=messages
        )
        if response.stop_reason == "end_turn":
            return "".join(b.text for b in response.content if hasattr(b, "text"))
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = execute_with_hooks(block.name, block.input)  # HOOKS APPLIED
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})
        if tool_results:
            messages.append({"role": "user", "content": tool_results})
    return "(cap)"

print("Running with PostToolUse hooks...\n")
hooked_result = run_agent_with_hooks("Get customer C001's record and order history. List all dates and amounts.")
print(f"Result:\n{hooked_result}")
print("\nAll dates should be YYYY-MM-DD, all amounts should be plain numbers -- guaranteed by the hook.")

---

## Phase 3: Your Turn -- PreToolCall Hook

Build a PreToolCall hook that blocks access to customer records based on user role.

In [ ]:
# =============================================================
# TODO: Build a PreToolCall hook for access control
# =============================================================
#
# Rules:
# - "support" role: can call get_customer and get_order_history
# - "analytics" role: can only call get_order_history (no customer PII)
# - Unknown roles: blocked from all tools
#
# The hook should:
# 1. Check the user_role against allowed tools
# 2. Return None if the call is allowed (proceed)
# 3. Return an error message string if blocked

ROLE_PERMISSIONS = {
    "support": ["get_customer", "get_order_history"],
    "analytics": ["get_order_history"],
}

def pre_tool_call_hook(tool_name, tool_input, user_role):
    """
    PreToolCall hook: checks if the user role is allowed to call this tool.
    
    Returns:
        None if allowed (proceed with tool call)
        Error message string if blocked
    """
    # TODO: Implement the access control logic
    # allowed_tools = ROLE_PERMISSIONS.get(user_role, [])
    # if tool_name not in allowed_tools:
    #     return json.dumps({"error": f"BLOCKED: Role '{user_role}' is not authorized to call '{tool_name}'"})
    # return None  # Allowed
    pass

print("TODO: Implement the pre_tool_call_hook function.")

In [ ]:
# Checker
print("=== ACCESS CONTROL HOOK TESTS ===")

tests = [
    ("support", "get_customer", None, "Support can access customer data"),
    ("support", "get_order_history", None, "Support can access order history"),
    ("analytics", "get_order_history", None, "Analytics can access order history"),
    ("analytics", "get_customer", "BLOCKED", "Analytics CANNOT access customer PII"),
    ("unknown", "get_customer", "BLOCKED", "Unknown role blocked from all tools"),
    ("unknown", "get_order_history", "BLOCKED", "Unknown role blocked from all tools"),
]

passed = 0
for role, tool, expected_contains, description in tests:
    result = pre_tool_call_hook(tool, {}, role)
    if expected_contains is None:
        ok = result is None
    else:
        ok = result is not None and expected_contains in str(result)
    
    status = "PASS" if ok else "FAIL"
    print(f"  [{status}] {description}")
    if ok:
        passed += 1

print(f"\n{passed}/{len(tests)} tests passed")
if passed == len(tests):
    print("PASSED -- Access control hook works correctly.")

---

## Phase 4: Stress Test

Verify hooks produce consistent results across multiple calls.

In [ ]:
# Run the normalization hook on all test data
print("=== NORMALIZATION CONSISTENCY TEST ===")

all_dates_normalized = True
all_money_normalized = True
iso_pattern = re.compile(r'^\d{4}-\d{2}-\d{2}$')
money_pattern = re.compile(r'^\d+\.\d{2}$')

for cid, record in CUSTOMER_RECORDS.items():
    raw = json.dumps(record)
    normalized = json.loads(post_tool_use_hook("get_customer", raw))
    
    print(f"\n  Customer {cid}:")
    for key, val in normalized.items():
        if any(d in key.lower() for d in ["date", "renewal", "purchase"]):
            is_iso = bool(iso_pattern.match(str(val)))
            print(f"    {key}: {val} {'[OK]' if is_iso else '[FAIL]'}")
            if not is_iso:
                all_dates_normalized = False
        elif any(m in key.lower() for m in ["balance", "total"]):
            is_plain = bool(money_pattern.match(str(val)))
            print(f"    {key}: {val} {'[OK]' if is_plain else '[FAIL]'}")
            if not is_plain:
                all_money_normalized = False

print(f"\nAll dates ISO 8601: {all_dates_normalized}")
print(f"All money plain decimal: {all_money_normalized}")
if all_dates_normalized and all_money_normalized:
    print("\nPASSED -- PostToolUse hooks produce 100% consistent normalization.")
    print("This is deterministic -- unlike prompt-based normalization.")

### Key Takeaways

1. **PostToolUse hooks normalize data before the model sees it.** The model gets clean, consistent data every time.
2. **PreToolCall hooks enforce access control.** They block unauthorized tool calls at the code level.
3. **Hooks are deterministic.** Unlike prompt instructions, they work 100% of the time because they execute in code.
4. **Use hooks for guarantees, prompts for guidance.** Normalization and security = hooks. Style and suggestions = prompts.